### Structured Output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [10]:
import os
from langchain.chat_models import init_chat_model

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

# model = init_chat_model("google_genai:gemini-2.5-flash")
model = init_chat_model(
    "llama3.2:3b",
    model_provider="ollama",
    # reasoning=False
)
model

ChatOllama(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, model='llama3.2:3b')

In [11]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")

In [12]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatOllama(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, model='llama3.2:3b'), kwargs={'format': {'properties': {'title': {'description': 'The title of the movie', 'title': 'Title', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'title': 'Year', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'title': 'Director', 'type': 'string'}, 'rating': {'description': 'The movies rating out of 10', 'title': 'Rating', 'type': 'number'}}, 'required': ['title', 'year', 'director', 'rating'], 'title': 'Movie', 'type': 'object'}, 'ls_structured_output_format': {'kwargs': {'method': 'json_schema'}, 'schema': <class '__main__.Movie'>}}, config={}, config_factories=[])
| PydanticOutputParser(pydantic_object=<class '__main__.Movie'>)

In [13]:
model.invoke("Provide details about movie Intersteller")

AIMessage(content="Interstellar is a 2014 science fiction film directed, written, and co-produced by Christopher Nolan. The film stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Michael Caine, Casey Affleck, Wes Bentley, Matt Damon, and John Lithgow.\n\n**Plot:**\n\nThe movie takes place in a dystopian future where Earth is facing an impending environmental disaster due to its inability to produce enough food. In response, a team of scientists, led by Professor Brand (Michael Caine), recruits a group of astronauts to travel through a wormhole in search of a new habitable planet for humanity.\n\nThe story follows Cooper (Matthew McConaughey), a former NASA pilot who is forced into retirement after an accident on the job. However, when he receives a message from Professor Brand's daughter, Murph (Jessica Chastain), he is recruited to join the mission. The crew consists of Dr. Romilly (David Gyasi), Dr. Mann (Matt Damon), and robots CASE (voiced by Bill Irwin) and TARS (voiced 

In [15]:
response = model_with_structure.invoke("Provide details about movie Intersteller")
response

Movie(title='Interstellar', year=2014, director='Christopher Nolan', rating=8.5)

### Message output alongside parsed structure

In [16]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)  

response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content='{"title": "Inception", "year": 2010, "director": "Christopher Nolan", "rating": 8.5}\n\n    \t\t\t  \t\t\t\t\t\t\t\t \t\t', additional_kwargs={}, response_metadata={'model': 'llama3.2:3b', 'created_at': '2026-06-26T14:06:18.4176818Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4061732800, 'load_duration': 267176900, 'prompt_eval_count': 32, 'prompt_eval_duration': 260909000, 'eval_count': 37, 'eval_duration': 3525550000, 'logprobs': None, 'model_name': 'llama3.2:3b', 'model_provider': 'ollama'}, id='lc_run--019f0440-a952-7913-a1d9-223025b095ee-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 32, 'output_tokens': 37, 'total_tokens': 69}),
 'parsed': Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.5),
 'parsing_error': None}

### Nested Structure

In [17]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito')], genres=['Action', 'Adventure', 'Thriller'], budget=1600000000.0)

### TypedDict
TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [18]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the movie avengers")
response

{'title': 'Avengers (2012)',
 'year': 2012,
 'director': 'Joss Whedon',
 'rating': 8.1}

In [19]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Avengers")
response

{'title': 'Avengers (2012)',
 'year': 2012,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'}],
 'genres': ['Action', 'Adventure', 'Science Fiction'],
 'budget': 220000000}

In [12]:
model.profile

{'name': 'Gemini 2.5 Flash',
 'release_date': '2025-03-20',
 'last_updated': '2025-06-05',
 'open_weights': False,
 'max_input_tokens': 1048576,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': True,
 'audio_inputs': True,
 'pdf_inputs': True,
 'video_inputs': True,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': True,
 'temperature': True,
 'image_url_inputs': True,
 'image_tool_message': True,
 'tool_choice': True}

### DataClasses
A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [ ]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model=model,
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='d59c3eb7-0319-4f52-a2e9-04a6d6e5c719'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2:3b', 'created_at': '2026-06-26T14:15:34.7432542Z', 'done': True, 'done_reason': 'stop', 'total_duration': 10403669200, 'load_duration': 3476169000, 'prompt_eval_count': 210, 'prompt_eval_duration': 4209264000, 'eval_count': 38, 'eval_duration': 2705292000, 'logprobs': None, 'model_name': 'llama3.2:3b', 'model_provider': 'ollama'}, id='lc_run--019f0449-0db2-7c31-8bf6-4bc05b4637da-0', tool_calls=[{'name': 'ContactInfo', 'args': {'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}, 'id': '87fcb371-5c1a-4d1a-b328-e98e48393dbf', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 210, 'output_tokens': 38, 'total_tokens': 248}),
  ToolMessage(content="Returning struct

In [21]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [22]:
## Typedict
from typing_extensions import TypedDict
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model="google_genai:gemini-2.5-flash",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]
# {'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [23]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


agent = create_agent(
    model="google_genai:gemini-2.5-flash",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')